# Prototyping Functions to Ingest Data

In [9]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from typing import List
from fredapi import Fred

## EIA Spot Prices

In [10]:
def fetch_eia_series_pet(series_ids=['RBRTE', 'RWTC'], # ID's of series we're pulling (Brent, WTI)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)


    URL_BASE = f"https://api.eia.gov/v2/petroleum/pri/spt/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide


def fetch_eia_series_gas(series_ids = ["RNGWHHD"], # ID's of series we're pulling (Henry Hub)
                         frequency= 'weekly', # Frequency of the spot prices, either daily, weekly, or monthly.
                         length = 5000): # Timespan that we're pulling, max 5000 weeks
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)
    

    URL_BASE = f"https://api.eia.gov/v2/natural-gas/pri/fut/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide

fetch_eia_series_gas()
    

series,RNGWHHD
period,
1997-01-10,3.79
1997-01-17,4.19
1997-01-24,2.98
1997-01-31,2.91
1997-02-07,2.53
...,...
2026-05-08,2.74
2026-05-15,2.86
2026-05-22,3.11


## FRED Series (DXY, VIX, 10Y - 2Y Spread)

In [11]:
def fetch_fred_series(series_ids=['T10Y2Y','VIXCLS','DTWEXBGS'], #ID's of the series we want to pull, here 10Y-2Y spread, VIX, and DXY
                      frequency = 'W-FRI'  # How frequent we want the samples to be. (day -> "D", weekly, on friday -> "W-FRI", monthly -> "M", yearly -> "Y"
                      ):
    
    load_dotenv()
    FRED_API_KEY = os.getenv('FRED_API_KEY')
    fred = Fred(api_key= FRED_API_KEY)
    series_dict = {}
    for id in series_ids: # Create a dict with key as ID and values as the corresponding series
        series_dict[id] = fred.get_series(series_id=id)

    df = pd.DataFrame(series_dict) # Turn dictionary into a DF
    df.index = pd.to_datetime(df.index) # Convert index dtype to datetime
    df.index.name = 'period' # Convert index name to 'period' to match the EIA information

    df = df.resample(rule=frequency).last() # Resample to keep only the days at the end of the week

    df = df.dropna() # Drop any NaNs

    return df
fetch_fred_series()

,T10Y2Y,VIXCLS,DTWEXBGS
period,,,
2006-01-06,0.02,11.00,100.0241
2006-01-13,0.02,11.23,99.9675
2006-01-20,0.00,14.56,99.9017
2006-01-27,0.01,11.97,99.6433
2006-02-03,-0.05,12.96,100.1180
...,...,...,...
2026-05-08,0.48,17.19,118.0392
2026-05-15,0.50,18.43,119.2825
2026-05-22,0.43,16.70,119.2868


In [12]:
def fetch_eia_stock(series_ids=['WCESTUS1'], # ID's of series we're pulling (Week-end US Crude Inventory)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)

    URL_BASE = f'https://api.eia.gov/v2/petroleum/stoc/wstk/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}'

    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide

fetch_eia_stock().head()

series,WCESTUS1
period,
1982-08-20,338764
1982-08-27,336138
1982-09-24,335586
1982-10-01,334786
1982-10-08,335260


In [13]:

def get_merged_df(eia_spot_pet_df=fetch_eia_series_pet(), eia_spot_gas_df=fetch_eia_series_gas(), eia_stock_df=fetch_eia_stock(),fred_df=fetch_fred_series()):
    # Get and merge eia and fred data
    df = pd.merge(left=eia_spot_pet_df, right=fred_df,left_index = True, right_index = True, how= 'inner')
    df = pd.merge(left=df, right=eia_stock_df, left_index = True, right_index = True, how = 'inner')
    df = pd.merge(left=df, right= eia_spot_gas_df, left_index = True, right_index = True, how = 'inner' )
    df = df.copy().reset_index().sort_values('period')

    # Get and merge opec meeting dates, and the time since a meeting (these are big events for the market)
        
    opec_dates = pd.read_csv('data/meetings.csv', parse_dates=['date']).sort_values('date')
    
    df = pd.merge_asof(df, opec_dates.assign(last_opec=opec_dates['date']),
                    left_on='period', right_on='date', direction='backward') #merge_asof used for timeseries data where the dates don't match, finds the closest date in the rightmost df and returns that value for all rows matches, here we use backward to get the most recent date, forward would get the soonest.
    
    df['days_since_opec'] = (df['period'] - df['last_opec']).dt.days
    df = df.drop(columns=['date', 'last_opec'])
    
    return df.set_index('period')

In [14]:
def compute_returns(df=get_merged_df(), method= 'log',series_ids= ['RBRTE','RWTC', 'RNGWHHD']):
    df = df.copy()
    for id in series_ids:
        df[id + '_log_return'] = np.log(df[id]/ df[id].shift(1))
    return df.dropna()
    
    


In [15]:
def rolling_z_scores(df, 
                     series_ids=['RBRTE_log_return','RWTC_log_return', 'RNGWHHD_log_return'],
                     window=63,
                     min_periods=1):
    df = df.copy()
    for id in series_ids:
      avg = df[id].rolling(window=window, min_periods=min_periods).mean()
      dev = df[id].rolling(window=window, min_periods=min_periods).std()
      df[id + '_rol_z_score'] = (df[id] - avg) / dev
    return df


        


In [16]:
def series_diff(df, series_ids=['WCESTUS1']):
    df = df.copy()
    for id in series_ids:
        df[id + '_w_change'] = df[id].diff()
    return df

series_diff(compute_returns(get_merged_df()))

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,days_since_opec,RBRTE_log_return,RWTC_log_return,RNGWHHD_log_return,WCESTUS1_w_change
period,,,,,,,,,,,,
2007-03-16,60.87,57.94,-0.03,16.79,97.0407,311926,6.86,1.0,0.008579,-0.049004,-0.064903,NaN
2007-03-23,61.09,58.26,0.02,12.95,96.5662,311080,6.91,8.0,0.003608,0.005508,0.007262,-846.0
2007-03-30,66.10,64.18,0.07,14.64,96.3183,315387,7.32,15.0,0.078821,0.096776,0.057641,4307.0
2007-04-06,68.55,64.82,0.01,13.23,96.2526,316219,7.55,22.0,0.036395,0.009923,0.030937,832.0
2007-04-13,68.20,62.58,0.00,12.20,95.7110,315225,7.83,29.0,-0.005119,-0.035168,0.036415,-994.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-08,105.88,102.28,0.48,17.19,118.0392,452876,2.74,159.0,-0.122097,-0.031660,0.029632,-4306.0
2026-05-15,110.53,105.10,0.50,18.43,119.2825,445013,2.86,166.0,0.042981,0.027198,0.042864,-7863.0
2026-05-22,110.61,105.32,0.43,16.70,119.2868,441686,3.11,173.0,0.000724,0.002091,0.083801,-3327.0


In [17]:
def lagged_features(df, series_ids= ['RBRTE','RWTC'], lags=[1,4,12]):
    df = df.copy()
    for id in series_ids:
        for lag in lags:
            df[id + f'{lag}_w_lag'] = df[id].shift(lag)
    return df

In [18]:
def rolling_vol(df, series_ids=['RBRTE_log_return','RWTC_log_return','RNGWHHD_log_return'], periods=[4,12]):
    df = df.copy()
    for id in series_ids:
        id = id.split("_")[0]
        for period in periods:
            df[id + f'_{period}_rol_vol'] = df[id].rolling(window=period,min_periods=1).std()
    return df
rolling_vol(compute_returns(get_merged_df()))




,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,days_since_opec,RBRTE_log_return,RWTC_log_return,RNGWHHD_log_return,RBRTE_4_rol_vol,RBRTE_12_rol_vol,RWTC_4_rol_vol,RWTC_12_rol_vol,RNGWHHD_4_rol_vol,RNGWHHD_12_rol_vol
period,,,,,,,,,,,,,,,,,
2007-03-16,60.87,57.94,-0.03,16.79,97.0407,311926,6.86,1.0,0.008579,-0.049004,-0.064903,NaN,NaN,NaN,NaN,NaN,NaN
2007-03-23,61.09,58.26,0.02,12.95,96.5662,311080,6.91,8.0,0.003608,0.005508,0.007262,0.155563,0.155563,0.226274,0.226274,0.035355,0.035355
2007-03-30,66.10,64.18,0.07,14.64,96.3183,315387,7.32,15.0,0.078821,0.096776,0.057641,2.958079,2.958079,3.513934,3.513934,0.252389,0.252389
2007-04-06,68.55,64.82,0.01,13.23,96.2526,316219,7.55,22.0,0.036395,0.009923,0.030937,3.798442,3.798442,3.706571,3.706571,0.331763,0.331763
2007-04-13,68.20,62.58,0.00,12.20,95.7110,315225,7.83,29.0,-0.005119,-0.035168,0.036415,3.438008,3.754673,2.954229,3.260626,0.388962,0.415126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-08,105.88,102.28,0.48,17.19,118.0392,452876,2.74,159.0,-0.122097,-0.031660,0.029632,5.961884,18.684594,5.567172,14.303059,0.047871,0.171630
2026-05-15,110.53,105.10,0.50,18.43,119.2825,445013,2.86,166.0,0.042981,0.027198,0.042864,5.834601,15.751444,4.674830,12.096976,0.086410,0.163677
2026-05-22,110.61,105.32,0.43,16.70,119.2868,441686,3.11,173.0,0.000724,0.002091,0.083801,5.753511,11.049915,1.537040,8.231440,0.196363,0.172423


In [19]:
def differentials(df, series_ids=['RBRTE', 'RWTC']):
    df = df.copy()
    pairings = [(x,y) for i, x in enumerate(series_ids) for y in series_ids[i+1:]] 
    for pair in pairings:
        id_a , id_b = pair 
        df[f'{id_a}_{id_b}_diff'] = df[id_a] - df[id_b]
    return df



In [20]:
series_ids=['RBRTE', 'RWTC', 'BDO']
combinations = []
for id in series_ids:
    for neg_id in series_ids[::-1]:
        pair = set((id, neg_id))
        if pair not in combinations and len(pair) == 2:
            combinations.append(pair)

print(combinations)


combinations = [(x,y) for i, x in enumerate(series_ids) for y in series_ids[i+1::]] # (x,y), get series id and entry index, then do all pairs of x and everything ahead of it. Produces one less each pass, so there is no wasted compute.

[{'BDO', 'RBRTE'}, {'RWTC', 'RBRTE'}, {'BDO', 'RWTC'}]


In [21]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import cloudscraper

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
}

scraper = cloudscraper.create_scraper(
    interpreter='nodejs',  # Use Node.js instead of native solvers for tougher challenges
    browser={
        'browser': 'chrome',
        'platform': 'linux',
        'desktop': True
    })

html = scraper.get(url='https://www.forexfactory.com/calendar/437-opec-meetings')
html_content = html.text
html_content

soup = BeautifulSoup(html_content,'html.parser')


calendar_event = soup.find_all('div', class_= 'flexBox noflex calendar-event-history')
calendar_event


[<div class="flexBox noflex calendar-event-history" data-ebase-id="437"> <div class="head"> <ul> <li class="left noborder nolink"> <strong>History</strong> </li> </ul> </div> <table class="calendar-event__history alternating calendarhistory"> <thead class="subhead"> <tr> <th class="calendarhistory__header calendarhistory__header--history">Expected Impact / Date</th> <th class="calendarhistory__header calendarhistory__header--history">Description</th> </tr> </thead> <tbody> <tr> <td class="calendarhistory__row nowrap calendarhistory__row--history"> <span class="icon icon--ff-impact-ora"></span> <a href="/calendar?day=jun7.2026#detail=149593">Jun 7, 2026</a> </td> <td class="calendarhistory__row calendarhistory__row--description"> <div><span class="darktext"></span></div> </td> </tr> <tr> <td class="calendarhistory__row nowrap calendarhistory__row--history"> <span class="icon icon--ff-impact-ora"></span> <a href="/calendar?day=nov30.2025#detail=145530">Nov 30, 2025</a> </td> <td class="c

In [22]:
df = pd.read_csv('data/meetings.csv', parse_dates= ['date'])
df_expanded = df.copy()
meeting_dates = df.copy()['date']
df_expanded = df.set_index('date').resample('D').ffill().reset_index()
df['opec_meeting'] = 1
df_expanded = df_expanded.merge(right=df, how='left', on ='date').fillna(0)

df_expanded['days_since_meeting'] = df_expanded["date"].apply(
        lambda d: (d - meeting_dates[meeting_dates <= d].max()).days
        if len(meeting_dates[meeting_dates <= d]) > 0 else None
    )

df_expanded

,date,opec_meeting,days_since_meeting
0,2007-03-15,1.0,0
1,2007-03-16,0.0,1
2,2007-03-17,0.0,2
3,2007-03-18,0.0,3
4,2007-03-19,0.0,4
...,...,...,...
7020,2026-06-03,0.0,185
7021,2026-06-04,0.0,186
7022,2026-06-05,0.0,187
7023,2026-06-06,0.0,188


In [26]:
def feature_pipeline(df=get_merged_df()):
    df = df.copy()
    df = compute_returns(df)
    df = series_diff(df)
    df = lagged_features(df)
    df = rolling_vol(df)
    df = differentials(df)
    df = rolling_z_scores(df)
    return df

feature_pipeline().columns

Index(['RBRTE', 'RWTC', 'T10Y2Y', 'VIXCLS', 'DTWEXBGS', 'WCESTUS1', 'RNGWHHD',
       'days_since_opec', 'RBRTE_log_return', 'RWTC_log_return',
       'RNGWHHD_log_return', 'WCESTUS1_w_change', 'RBRTE1_w_lag',
       'RBRTE4_w_lag', 'RBRTE12_w_lag', 'RWTC1_w_lag', 'RWTC4_w_lag',
       'RWTC12_w_lag', 'RBRTE_4_rol_vol', 'RBRTE_12_rol_vol', 'RWTC_4_rol_vol',
       'RWTC_12_rol_vol', 'RNGWHHD_4_rol_vol', 'RNGWHHD_12_rol_vol',
       'RBRTE_RWTC_diff', 'RBRTE_log_return_rol_z_score',
       'RWTC_log_return_rol_z_score', 'RNGWHHD_log_return_rol_z_score'],
      dtype='str')

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_test(df,series_ids=['RBRTE_log_return', 'RWTC_log_return',
       'RNGWHHD_log_return','WCESTUS1_w_change', 'T10Y2Y', 'VIXCLS', 'DTWEXBGS']):
    
    results = {}
    p_values = {}
    for id in series_ids:
        print(f"Results of Dickey-Fuller Test on {id}:")
        dftest = adfuller(df[id], autolag="AIC")
        dfoutput = pd.Series(
            dftest[0:4],
            index=[
                "Test Statistic",
                "p-value",
                "#Lags Used",
                "Number of Observations Used",
            ],
        )
        for key, value in dftest[4].items():
            dfoutput["Critical Value (%s)" % key] = value
        print(dfoutput)
        p_values[id] = dfoutput['p_value']
        results[id] = dfoutput

    return results

adf_test(feature_pipeline(), ['RBRTE_log_return'])


Results of Dickey-Fuller Test on RBRTE_log_return:
Test Statistic                -1.064802e+01
p-value                        4.740047e-19
#Lags Used                     8.000000e+00
Number of Observations Used    9.950000e+02
Critical Value (1%)           -3.436939e+00
Critical Value (5%)           -2.864449e+00
Critical Value (10%)          -2.568319e+00
dtype: float64


{'RBRTE_log_return': Test Statistic                -1.064802e+01
 p-value                        4.740047e-19
 #Lags Used                     8.000000e+00
 Number of Observations Used    9.950000e+02
 Critical Value (1%)           -3.436939e+00
 Critical Value (5%)           -2.864449e+00
 Critical Value (10%)          -2.568319e+00
 dtype: float64}